In [2]:

from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import RandomForestRegressor

import joblib

In [ ]:
#Features:

#job_category
#years_experience
#skill_count
#company_location 

#Target:
#salary_usd

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../datasets/final_clean_jobs_dataset.csv")
df.head()

features = [
    "job_category",
    "years_experience",
    "skill_count",
    "company_location"
]

x = df[features]
y = df["salary_usd"] # target value 


In [ ]:
#Train Data set 
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2, # 20% test and 80% train
    random_state=42 
)

In [5]:
#Preprocessing
categorical_features = [
    "job_category",
    "company_location"
]
numeric_features = [
    "years_experience",
    "skill_count"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)


In [6]:
#Model 1 — Linear Regression
import joblib

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model",LinearRegression())
])
#Train 
lr_pipeline.fit(
    X_train,
    y_train
)
#predict
pred_lr = lr_pipeline.predict(X_test)

#Evaluate

print("Linear Regression")

print(
    "MAE:",
    mean_absolute_error(y_test,pred_lr)
)

#save model 

joblib.dump(
    lr_pipeline,
    "../models/linear_regression.pkl"
)

Linear Regression
MAE: 21158.619430087903


['../models/linear_regression.pkl']

In [7]:
# Model 2 — Random Forest

from sklearn.utils._metadata_requests import _raise_for_unsupported_routing
rf_pipeline = Pipeline([
    ("preprocessor",preprocessor),
    (
        "model",
        RandomForestRegressor(
            n_estimators=200,
            random_state=42
        )
    )
])

#train

rf_pipeline.fit(
    X_train,
    y_train
)

#predict

pred_rf = rf_pipeline.predict(X_test)

# Evaluate
print("Random Forest")

print(
    "MAE:",
    mean_absolute_error(y_test, pred_rf)
)

print(
    "R2:",
    r2_score(y_test, pred_rf)
)


#save 

joblib.dump(
    rf_pipeline,
    "../models/random_forest.pkl"
)



Random Forest
MAE: 18825.543828798203
R2: 0.8010658274839044


['../models/random_forest.pkl']

In [8]:
# Model 3 — XGBoost

xgb_pipeline = Pipeline([
    ("preprocessor",preprocessor),
    (
        "model",
        XGBRegressor(
            n_estimators = 300,
            learning_rate = 0.05,
            max_depth = 6,
            random_state = 42
        )
    )
])

#Train

xgb_pipeline.fit(
    X_train,
    y_train
)

#Predict

pred_xgb = xgb_pipeline.predict(X_test)

#Evaluate
print("XGBoost")

print(
    "MAE:",
    mean_absolute_error(y_test, pred_xgb)
)

print(
    "R2:",
    r2_score(y_test, pred_xgb)
)

#save

joblib.dump(
    xgb_pipeline,
    "../models/xgboost.pkl"
)


XGBoost
MAE: 17164.9140625
R2: 0.8360572457313538


['../models/xgboost.pkl']

In [13]:
#--------Compare Models--------
from operator import index

#create data frame
comparison = pd.DataFrame({

    "Model" : [
        "Linear Regression",
        "Random Forest",
        "XGBoost"
    ],

    "R2" : [
        r2_score(y_test,pred_lr),
        r2_score(y_test,pred_rf),
        r2_score(y_test,pred_xgb)
    ],

# mean absolute error
    "MAE": [
        mean_absolute_error(y_test, pred_lr),
        mean_absolute_error(y_test, pred_rf),
        mean_absolute_error(y_test, pred_xgb)
    ]

}) 

comparison

# save to data frame in to CSV

comparison.to_csv(
    "../outputs/model_comparison.csv",
    index=False
)

In [15]:
#Test Prediction

sample = pd.DataFrame({
    "job_category": ["Machine Learning"],
    "years_experience": [5],
    "skill_count": [4],
    "company_location": ["United States"]
})

#prdeict

salary = xgb_pipeline.predict(sample)

print("Expected Salary:", salary[0])

Expected Salary: 152189.7


In [10]:
import os

print(os.getcwd())

c:\Users\UMAR\Documents\GitHub\ai-interview-job-market-platform\job-market-engine\notebooks
